# 02 — Spatial Exploratory Data Analysis
**Where Should I Buy? · GeoAI Course**

Goals:
- Explore spatial distribution of transactions per neighborhood
- Analyze price gradients across Gush Dan
- Visualize transit stop density
- Identify park accessibility gaps

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from src.data.loaders import load_transactions, load_neighborhoods, load_transit_stops, load_parks
from src.data.preprocessing import (
    preprocess_transactions, preprocess_neighborhoods,
    preprocess_transit_stops, preprocess_parks
)
from src.geo.spatial_join import aggregate_transactions_by_neighborhood

# Load and preprocess
transactions  = preprocess_transactions(load_transactions())
neighborhoods = preprocess_neighborhoods(load_neighborhoods())
transit_stops = preprocess_transit_stops(load_transit_stops())
parks         = preprocess_parks(load_parks())

print('Data loaded. Shapes:', transactions.shape, neighborhoods.shape)
print('CRS:', transactions.crs)

## 1. Transaction Price Distribution by Neighborhood

In [ ]:
stats = aggregate_transactions_by_neighborhood(transactions, 'neighborhood')
stats_sorted = stats.sort_values('avg_price', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(stats_sorted['name'], stats_sorted['avg_price'] / 1_000_000,
               color='steelblue', alpha=0.8)
ax.set_xlabel('Average Transaction Price (₪M)')
ax.set_title('Average Apartment Price by Municipality — Gush Dan', fontweight='bold')
ax.axvline(stats_sorted['avg_price'].mean() / 1_000_000, color='red',
           linestyle='--', label='Area Average')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Choropleth: Average Price per Municipality

In [ ]:
# Join stats to polygons
hood_map = neighborhoods.merge(stats[['name', 'avg_price', 'transaction_count']], on='name', how='left')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Price choropleth
hood_map.plot(column='avg_price', ax=axes[0], legend=True,
              cmap='RdYlGn_r', missing_kwds={'color': 'lightgray'},
              legend_kwds={'label': 'Avg Price (₪)', 'shrink': 0.8})
axes[0].set_title('Average Transaction Price', fontweight='bold')

# Transaction density
hood_map.plot(column='transaction_count', ax=axes[1], legend=True,
              cmap='Blues', missing_kwds={'color': 'lightgray'},
              legend_kwds={'label': 'Transactions', 'shrink': 0.8})
axes[1].set_title('Transaction Density', fontweight='bold')

# Add neighborhood labels
for _, row in hood_map.iterrows():
    if pd.notna(row.geometry):
        c = row.geometry.centroid
        for ax in axes:
            ax.annotate(row['name'].split('-')[0], xy=(c.x, c.y),
                       ha='center', fontsize=7, color='navy')

plt.tight_layout()
plt.show()

## 3. Transit Stop Density

In [ ]:
print('Transit stop types:')
print(transit_stops['stop_type'].value_counts())

fig, ax = plt.subplots(figsize=(10, 8))
neighborhoods.plot(ax=ax, color='lightyellow', edgecolor='gray', alpha=0.7)

colors = {'bus': 'blue', 'rail': 'red', 'light_rail': 'purple', 'unknown': 'gray'}
for stop_type, group in transit_stops.groupby('stop_type'):
    group.plot(ax=ax, color=colors.get(stop_type, 'gray'),
               markersize=4, alpha=0.6, label=stop_type.title())

ax.set_title('Transit Stop Distribution — Gush Dan', fontweight='bold')
ax.legend(title='Stop Type')
plt.tight_layout()
plt.show()

## 4. Key Statistics Summary

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Transactions:   {len(transactions):,}')
print(f'Neighborhoods:  {len(neighborhoods)}')
print(f'Transit Stops:  {len(transit_stops):,}')
print(f'Parks:          {len(parks):,}')
print()
print('=== PRICE STATISTICS ===')
print(transactions['transaction_price'].describe().apply(lambda x: f'₪{x:,.0f}'))